# Pipeline truy vấn pháp luật (baseline và cải tiến)

Notebook này triển khai pipeline truy xuất theo mô tả trong `mota.md`:

- **Baseline**: truy vấn lexical — TF-IDF + cosine trên toàn bộ corpus (không lọc chủ đề).
- **Cải tiến**: hai giai đoạn — **TextCNN** dự đoán chủ đề câu hỏi (top-2), lọc corpus theo nhãn chủ đề (`macro_domain`), sau đó **Siamese BiLSTM** mã hóa câu hỏi và đoạn văn rồi xếp hạng bằng cosine similarity.

Dữ liệu: corpus `vnlegal_corpus` và QA `vnlegal_qa` (join `passage_id`). Đánh giá so với đoạn vàng: **Recall@K**, **Precision@K**, **nDCG@K**, **MRR**, **MAP** (một đoạn vàng/query thì MAP = MRR), **MR_hit / MedR_hit** khi tìm thấy trong top-K, và **phân tích theo `difficulty` / `question_type`**.

**Phụ thuộc gợi ý:** `torch`, `datasets`, `scikit-learn`, `numpy`, `tqdm`, `underthesea` (tách từ tiếng Việt như trong mô tả đề tài).

In [ ]:
from __future__ import annotations

import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

try:
    from underthesea import word_tokenize
except ImportError:
    word_tokenize = None


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        arrow = p / "data" / "raw" / "vnlegal_corpus" / "data-00000-of-00001.arrow"
        if arrow.is_file():
            return p
        p = p.parent
    return Path.cwd().resolve()


REPO_ROOT = find_repo_root()
CORPUS_ARROW = REPO_ROOT / "data" / "raw" / "vnlegal_corpus" / "data-00000-of-00001.arrow"
QA_ARROW = REPO_ROOT / "data" / "raw" / "vnlegal_qa" / "data-00000-of-00001.arrow"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Giới hạn mẫu để chạy nhanh trên máy cấu hình vừa phải; tăng khi có GPU/thời gian.
MAX_QA_TRAIN = 4000
MAX_QA_EVAL = 800
TOPIC_TOPK = 2
RETRIEVE_K = 10
VOCAB_SIZE = 30_000
EMBED_DIM = 200
CNN_MAX_LEN = 64
DOC_MAX_LEN = 256
TRIPLET_MARGIN = 0.3
STRATIFY_MIN_GROUP = 20  # tối thiểu mẫu mỗi nhóm khi báo metric theo difficulty / question_type

print("REPO_ROOT:", REPO_ROOT)
print("DEVICE:", DEVICE)
print("Corpus exists:", CORPUS_ARROW.is_file(), "QA exists:", QA_ARROW.is_file())

In [ ]:
# underthesea.word_tokenize trả list token
def tokenize_vi_fixed(text: str) -> list[str]:
    text = (text or "").strip().lower()
    if not text:
        return []
    if word_tokenize is not None:
        toks = word_tokenize(text)
        if isinstance(toks, str):
            return toks.split()
        return list(toks)
    return text.split()


corpus = Dataset.from_file(str(CORPUS_ARROW))
qa = Dataset.from_file(str(QA_ARROW))

passage_list = corpus["passage_id"]
passage_to_idx = {pid: i for i, pid in enumerate(passage_list)}
num_passages = len(passage_list)

articles = corpus["article_content"]
macro_domains = corpus["macro_domain"]

domain_names = sorted(set(macro_domains))
domain_to_id = {d: i for i, d in enumerate(domain_names)}
num_topics = len(domain_names)

print("Số đoạn corpus:", num_passages, "| Số QA:", len(qa), "| Số chủ đề (macro_domain):", num_topics)


In [ ]:
def build_qa_splits(qa_ds: Dataset, seed: int = SEED):
    rng = random.Random(seed)
    indices = list(range(len(qa_ds)))
    rng.shuffle(indices)
    n_train = min(MAX_QA_TRAIN, int(0.85 * len(indices)))
    n_eval = min(MAX_QA_EVAL, len(indices) - n_train)
    train_idx = set(indices[:n_train])
    eval_idx = indices[n_train : n_train + n_eval]
    return train_idx, eval_idx


train_idx_set, eval_indices = build_qa_splits(qa)
eval_rows = qa.select(eval_indices)
print("Train QA (ước lượng):", len(train_idx_set), "Eval QA:", len(eval_rows))

## Baseline: TF-IDF + cosine (toàn corpus)

Không lọc chủ đề; phù hợp làm đường chuẩn so sánh với pipeline hai giai đoạn.

**Metrics (một đoạn vàng / query):** Recall@K và Precision@K; nDCG@K (gain nhị phân, chuẩn hóa với đoạn vàng ở hạng 1); MRR; MAP (trùng MRR khi chỉ một đoạn đúng); MR_hit / MedR_hit — trung bình / trung vị hạng khi đoạn vàng nằm trong top-K đánh giá.

In [ ]:
def join_tokens(tokens: list[str]) -> str:
    return " ".join(tokens)


corpus_tokens = [join_tokens(tokenize_vi_fixed(a)) for a in tqdm(articles, desc="Tokenize corpus")]

tfidf = TfidfVectorizer(max_features=80_000, ngram_range=(1, 2), min_df=2)
doc_tfidf = tfidf.fit_transform(corpus_tokens)
doc_tfidf_norm = normalize(doc_tfidf, norm="l2", axis=1)


def retrieve_baseline_tfidf(question: str, k: int = RETRIEVE_K):
    qvec = tfidf.transform([join_tokens(tokenize_vi_fixed(question))])
    qvec = normalize(qvec, norm="l2", axis=1)
    sims = (doc_tfidf_norm @ qvec.T).toarray().ravel()
    top = np.argpartition(-sims, kth=min(k, len(sims) - 1))[:k]
    top = top[np.argsort(-sims[top])]
    return top.tolist(), sims[top].tolist()


def eval_retrieval(rank_fn, eval_rows_iter, ks=(1, 5, 10), desc="Eval"):
    """Mỗi query một đoạn vàng (passage_id). Trả về các metric IR chuẩn."""
    k_max = max(ks)
    recalls = {kk: [] for kk in ks}
    precs = {kk: [] for kk in ks}
    ndcgs = {kk: [] for kk in ks}
    rr, aps = [], []
    ranks_hit = []
    n_skip = 0

    for row in tqdm(eval_rows_iter, desc=desc):
        gold = row["passage_id"]
        if gold not in passage_to_idx:
            n_skip += 1
            continue
        idxs, _ = rank_fn(row["question"], k_max)
        pred = [passage_list[i] for i in idxs]
        rank = None
        for i, pid in enumerate(pred[:k_max]):
            if pid == gold:
                rank = i + 1
                break
        if rank is not None:
            ranks_hit.append(rank)

        for kk in ks:
            hit = rank is not None and rank <= kk
            recalls[kk].append(1.0 if hit else 0.0)
            precs[kk].append((1.0 / kk) if hit else 0.0)
            if hit:
                dcg = 1.0 / np.log2(rank + 1)
                idcg = 1.0 / np.log2(2.0)
                ndcgs[kk].append(dcg / idcg)
            else:
                ndcgs[kk].append(0.0)

        if rank is not None:
            rr.append(1.0 / rank)
            aps.append(1.0 / rank)
        else:
            rr.append(0.0)
            aps.append(0.0)

    n = len(recalls[ks[0]])
    out = {f"Recall@{kk}": float(np.mean(recalls[kk])) for kk in ks}
    out.update({f"Precision@{kk}": float(np.mean(precs[kk])) for kk in ks})
    out.update({f"nDCG@{kk}": float(np.mean(ndcgs[kk])) for kk in ks})
    out["MRR"] = float(np.mean(rr)) if n else 0.0
    out["MAP"] = float(np.mean(aps)) if n else 0.0
    out["n_queries"] = float(n)
    out["n_skipped_no_passage"] = float(n_skip)
    if ranks_hit:
        out["MR_hit"] = float(np.mean(ranks_hit))
        out["MedR_hit"] = float(np.median(ranks_hit))
    else:
        out["MR_hit"] = float("nan")
        out["MedR_hit"] = float("nan")
    return out


def eval_retrieval_stratified(rank_fn, eval_dataset: Dataset, ks=(1, 5, 10), min_group: int = 20):
    """Theo difficulty / question_type (mota.md: phân tích theo nhóm)."""
    rows = [r for r in eval_dataset if r["passage_id"] in passage_to_idx]
    out = {}
    for field in ("difficulty", "question_type"):
        groups = {}
        for r in rows:
            groups.setdefault(r.get(field) or "unknown", []).append(r)
        out[field] = {}
        for name in sorted(groups):
            g = groups[name]
            if len(g) < min_group:
                continue
            out[field][name] = eval_retrieval(
                rank_fn, g, ks=ks, desc=f"Eval {field}={name}",
            )
    return out


def print_stratified(title: str, strat: dict):
    print(f"\n=== {title} (stratified) ===")
    for field, groups in strat.items():
        print(f"\n-- {field} --")
        for gname, m in groups.items():
            print(f"  [{gname}] n={int(m.get('n_queries', 0))}", end="")
            for key in sorted(k for k in m if k.startswith(("Recall", "Precision", "nDCG", "MRR", "MAP"))):
                print(f"  {key}={m[key]:.4f}", end="")
            print()


baseline_metrics = eval_retrieval(
    lambda q, k: retrieve_baseline_tfidf(q, k),
    eval_rows,
)
print("Baseline TF-IDF:", baseline_metrics)

## Cải tiến: TextCNN (top-2 chủ đề) + Siamese BiLSTM

Cấu hình tham chiếu `mota.md`: filter CNN k=[2,3,4], 128 filters mỗi nhánh; BiLSTM 2 lớp hidden 256, mean pooling có mask, L2 norm, triplet margin 0.3.

Nhãn chủ đề lấy từ trường **`macro_domain`** của corpus (ứng với đoạn vàng của từng câu hỏi).

In [ ]:
PAD, UNK = "<pad>", "<unk>"


def build_vocab(texts_tokenized: list[list[str]], max_vocab: int = VOCAB_SIZE):
    cnt = Counter()
    for toks in texts_tokenized:
        cnt.update(toks)
    most = [w for w, _ in cnt.most_common(max_vocab - 2)]
    itos = [PAD, UNK] + most
    stoi = {t: i for i, t in enumerate(itos)}
    return stoi, itos


def encode_tokens(stoi: dict[str, int], toks: list[str], max_len: int) -> tuple[list[int], list[float]]:
    unk = stoi[UNK]
    ids = [stoi.get(t, unk) for t in toks[:max_len]]
    pad_id = stoi[PAD]
    length = len(ids)
    if length < max_len:
        ids = ids + [pad_id] * (max_len - length)
    mask = [1.0] * length + [0.0] * (max_len - length)
    return ids, mask


train_questions: list[str] = []
train_domains: list[int] = []
for i, row in enumerate(qa):
    if i not in train_idx_set:
        continue
    pid = row["passage_id"]
    if pid not in passage_to_idx:
        continue
    dom = macro_domains[passage_to_idx[pid]]
    train_questions.append(row["question"])
    train_domains.append(domain_to_id[dom])

train_q_toks = [tokenize_vi_fixed(q) for q in tqdm(train_questions, desc="Tokenize train questions")]

# Vocab từ câu hỏi train + một phần corpus (tránh quá nặng)
sample_doc_idx = np.random.default_rng(SEED).choice(num_passages, size=min(2000, num_passages), replace=False)
extra_toks = [tokenize_vi_fixed(articles[i]) for i in tqdm(sample_doc_idx, desc="Tokenize sample docs")]
all_for_vocab = train_q_toks + extra_toks
stoi, itos = build_vocab(all_for_vocab, VOCAB_SIZE)
print("Vocab size:", len(itos))

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, num_classes: int, filter_sizes=(2, 3, 4), num_filters=128, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in filter_sizes])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L)
        emb = self.embedding(x).transpose(1, 2)  # (B, E, L)
        pooled = []
        for conv in self.convs:
            h = F.relu(conv(emb))
            h = h.max(dim=2).values
            pooled.append(h)
        z = torch.cat(pooled, dim=1)
        z = self.dropout(z)
        return self.fc(z)


class SiameseBiLSTM(nn.Module):
    def __init__(self, embedding: nn.Embedding, hidden_size: int = 256, num_layers: int = 2, dropout: float = 0.3):
        super().__init__()
        self.embedding = embedding
        self.lstm = nn.LSTM(
            embedding.embedding_dim,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.hidden_size = hidden_size

    def encode(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x)
        emb = self.drop(emb)
        out, _ = self.lstm(emb)  # (B, L, 2H)
        m = mask.unsqueeze(-1)
        summed = (out * m).sum(dim=1)
        denom = m.sum(dim=1).clamp(min=1e-6)
        v = summed / denom
        return F.normalize(v, p=2, dim=1)

    def forward(self, q, q_mask, d, d_mask):
        return self.encode(q, q_mask), self.encode(d, d_mask)

In [ ]:
def batch_encode_questions(questions: list[str], max_len: int) -> tuple[torch.Tensor, torch.Tensor]:
    ids, masks = [], []
    for q in questions:
        t = tokenize_vi_fixed(q)
        i, m = encode_tokens(stoi, t, max_len)
        ids.append(i)
        masks.append(m)
    return torch.tensor(ids, dtype=torch.long), torch.tensor(masks, dtype=torch.float32)


def batch_encode_docs(doc_texts: list[str], max_len: int) -> tuple[torch.Tensor, torch.Tensor]:
    ids, masks = [], []
    for doc in doc_texts:
        t = tokenize_vi_fixed(doc)
        i, m = encode_tokens(stoi, t, max_len)
        ids.append(i)
        masks.append(m)
    return torch.tensor(ids, dtype=torch.long), torch.tensor(masks, dtype=torch.float32)


topic_model = TextCNN(len(itos), EMBED_DIM, num_topics).to(DEVICE)
opt_cnn = torch.optim.Adam(topic_model.parameters(), lr=1e-3)

y_tensor = torch.tensor(train_domains, dtype=torch.long)
X_ids, X_mask = batch_encode_questions(train_questions, CNN_MAX_LEN)
cls_counts = np.bincount(train_domains, minlength=num_topics)
w = 1.0 / np.sqrt(cls_counts + 1e-6)
w = torch.tensor(w / w.mean(), dtype=torch.float32, device=DEVICE)
crit_cnn = nn.CrossEntropyLoss(weight=w)

ds_cnn = TensorDataset(X_ids, y_tensor)
dl_cnn = DataLoader(ds_cnn, batch_size=64, shuffle=True)

n_epochs_cnn = 3
topic_model.train()
for ep in range(n_epochs_cnn):
    total, n = 0.0, 0
    for xb, yb in tqdm(dl_cnn, desc=f"TextCNN epoch {ep+1}"):
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        opt_cnn.zero_grad()
        logits = topic_model(xb)
        loss = crit_cnn(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(topic_model.parameters(), 1.0)
        opt_cnn.step()
        total += loss.item() * xb.size(0)
        n += xb.size(0)
    print(f"  CNN loss: {total / n:.4f}")

topic_model.eval()

In [ ]:
def predict_topic_topk(model: TextCNN, question: str, k: int = TOPIC_TOPK) -> list[int]:
    xb, _ = batch_encode_questions([question], CNN_MAX_LEN)
    xb = xb.to(DEVICE)
    with torch.no_grad():
        logits = model(xb)
        probs = F.softmax(logits, dim=-1)[0]
        topk = torch.topk(probs, k=min(k, probs.numel()))
    return topk.indices.tolist()


passage_domain_ids = torch.tensor([domain_to_id[d] for d in macro_domains], dtype=torch.long, device=DEVICE)
passage_domain_ids_np = passage_domain_ids.cpu().numpy()


def candidate_indices_for_topics(topic_ids: list[int]) -> np.ndarray:
    mask = np.zeros(num_passages, dtype=bool)
    for tid in topic_ids:
        mask |= passage_domain_ids_np == tid
    idxs = np.where(mask)[0]
    if idxs.size == 0:
        return np.arange(num_passages)
    return idxs

In [ ]:
shared_embedding = topic_model.embedding
siamese = SiameseBiLSTM(shared_embedding).to(DEVICE)
opt_s = torch.optim.Adam(siamese.parameters(), lr=5e-4)


# Danh sách train có passage_id (anchor/positive cho triplet)
train_rows = []
for i, row in enumerate(qa):
    if i not in train_idx_set:
        continue
    if row["passage_id"] not in passage_to_idx:
        continue
    train_rows.append(row)


def sample_triplet():
    r = random.choice(train_rows)
    q = r["question"]
    pos_doc = articles[passage_to_idx[r["passage_id"]]]
    dom = macro_domains[passage_to_idx[r["passage_id"]]]
    # Negative: cùng chủ đề khác đoạn (same-topic hard)
    cand = [j for j in range(num_passages) if macro_domains[j] == dom and passage_list[j] != r["passage_id"]]
    if not cand:
        cand = [j for j in range(num_passages) if passage_list[j] != r["passage_id"]]
    nj = random.choice(cand)
    neg_doc = articles[nj]
    return q, pos_doc, neg_doc


n_steps = 400
siamese.train()
for step in tqdm(range(n_steps), desc="Siamese triplet steps"):
    qs, ps, ns = zip(*(sample_triplet() for _ in range(32)))
    q_ids, q_m = batch_encode_questions(list(qs), CNN_MAX_LEN)
    p_ids, p_m = batch_encode_docs(list(ps), DOC_MAX_LEN)
    n_ids, n_m = batch_encode_docs(list(ns), DOC_MAX_LEN)
    q_ids, q_m = q_ids.to(DEVICE), q_m.to(DEVICE)
    p_ids, p_m = p_ids.to(DEVICE), p_m.to(DEVICE)
    n_ids, n_m = n_ids.to(DEVICE), n_m.to(DEVICE)
    opt_s.zero_grad()
    vq = siamese.encode(q_ids, q_m)
    vp = siamese.encode(p_ids, p_m)
    vn = siamese.encode(n_ids, n_m)
    sim_pos = (vq * vp).sum(dim=1)
    sim_neg = (vq * vn).sum(dim=1)
    loss = torch.relu(TRIPLET_MARGIN - sim_pos + sim_neg).mean()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(siamese.parameters(), 1.0)
    opt_s.step()
    if step % 100 == 0:
        print(f"  step {step} triplet loss: {loss.item():.4f}")

siamese.eval()

In [ ]:
@torch.no_grad()
def encode_all_passages(model: SiameseBiLSTM, batch_size: int = 64) -> np.ndarray:
    vecs = []
    model.eval()
    for start in tqdm(range(0, num_passages, batch_size), desc="Encode corpus"):
        batch = articles[start : start + batch_size]
        x, m = batch_encode_docs(batch, DOC_MAX_LEN)
        x, m = x.to(DEVICE), m.to(DEVICE)
        v = model.encode(x, m)
        vecs.append(v.cpu().numpy())
    return np.concatenate(vecs, axis=0)


doc_emb_all = encode_all_passages(siamese)


@torch.no_grad()
def encode_query_vec(question: str) -> np.ndarray:
    x, m = batch_encode_questions([question], CNN_MAX_LEN)
    x, m = x.to(DEVICE), m.to(DEVICE)
    v = siamese.encode(x, m)
    return v.cpu().numpy()


def retrieve_improved(question: str, k: int = RETRIEVE_K):
    topics = predict_topic_topk(topic_model, question, TOPIC_TOPK)
    cand_idx = candidate_indices_for_topics(topics)
    qv = encode_query_vec(question)  # (1, D)
    sub = doc_emb_all[cand_idx]
    sims = (sub @ qv.T).ravel()
    order = np.argsort(-sims)[:k]
    top_global = cand_idx[order].tolist()
    scores = sims[order].tolist()
    return top_global, scores


improved_metrics = eval_retrieval(
    lambda q, kk: retrieve_improved(q, kk),
    eval_rows,
)
print("Improved (TextCNN filter + Siamese):", improved_metrics)
print("Baseline TF-IDF:", baseline_metrics)

In [ ]:
def show_comparison():
    keys = sorted(set(baseline_metrics) | set(improved_metrics))
    w = max(12, max(len(k) for k in keys) + 2)
    hdr = f"{'Metric':<{w}} {'Baseline':>12} {'Improved':>12}"
    print(hdr)
    print("-" * len(hdr))
    for k in keys:
        b = baseline_metrics.get(k)
        im = improved_metrics.get(k)
        if isinstance(b, float) and np.isnan(b):
            bv = "nan"
        elif isinstance(b, (int, float)):
            bv = f"{b:.4f}"
        else:
            bv = str(b)
        if isinstance(im, float) and np.isnan(im):
            iv = "nan"
        elif isinstance(im, (int, float)):
            iv = f"{im:.4f}"
        else:
            iv = str(im)
        print(f"{k:<{w}} {bv:>12} {iv:>12}")


show_comparison()

baseline_strat = eval_retrieval_stratified(
    lambda q, kk: retrieve_baseline_tfidf(q, kk), eval_rows, min_group=STRATIFY_MIN_GROUP,
)
improved_strat = eval_retrieval_stratified(
    lambda q, kk: retrieve_improved(q, kk), eval_rows, min_group=STRATIFY_MIN_GROUP,
)
print_stratified("Baseline TF-IDF", baseline_strat)
print_stratified("Improved pipeline", improved_strat)

demo_q = eval_rows[0]["question"]
print("\nDemo question:", demo_q[:200], "...")
bi_idx, bi_sc = retrieve_baseline_tfidf(demo_q, 3)
im_idx, im_sc = retrieve_improved(demo_q, 3)
print("\nBaseline top-3 passage_id:", [passage_list[i] for i in bi_idx])
print("Improved top-3 passage_id:", [passage_list[i] for i in im_idx])
print("Gold:", eval_rows[0]["passage_id"])


### Ghi chú triển khai

- **fastText tiếng Việt**: có thể khởi tạo `nn.Embedding` từ file `.vec` và đặt `topic_model.embedding.weight.data` trước khi huấn luyện CNN (theo `mota.md`).
- Tăng `MAX_QA_TRAIN`, `n_steps` Siamese và `n_epochs_cnn` để kết quả gần với thiết lập đầy đủ trong báo cáo.
- Nếu TextCNN lọc sai chủ đề, có thể **fallback**: khi tập ứng viên sau lọc quá nhỏ hoặc score thấp, mở rộng thêm chủ đề hoặc truy vấn lại trên toàn corpus (chiến lược an toàn cho recall).